# 3-D cell thermal twin of the BYD FC4680 (paper 1.2)
Runs the verification and a micro-short case on a coarse grid (a few minutes on Colab's CPU). The full study: `python thermal3d.py short` (about 40 min on 2 cores).

In [ ]:
!git clone -q --depth 1 -b paper-1.2-3d-thermal-model https://github.com/RemusTeodorescu/4680-cell-models.git
%cd 4680-cell-models/3D-thermal-model-of-LFP-4680-cell-for-hotspot-scanning/cell_twin
!pip -q install pyamg

In [ ]:
!python geom3d.py | head -30

In [ ]:
!python thermal3d.py verify

## A 1 W short at mid-radius: response inside and on the can

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from geom3d import Cell, build_grid
from thermal3d import BC, assemble, Solver, box_source, surface_masks, field_stats, schedule_to, short_locations
c = Cell(); g = build_grid(c, refine=0.75)
K, C, Gb = assemble(g, BC(22.4, 22.4, 22.4)); sol = Solver(K, C)
m = surface_masks(g); out = {}
for name in ("core", "mid_radius", "at_cap_tab"):
    Q = box_source(g, 1.0, *short_locations(c)[name]); t, a, b = [], [], []
    sol.run(Q, schedule_to(600), callback=lambda tt, T: (t.append(tt), a.append(field_stats(g, T, m)['max_internal']), b.append(field_stats(g, T, m)['max_side'])))
    out[name] = (t, a, b)
for name, (t, a, b) in out.items():
    l, = plt.semilogx(t, a, label=name + ' inside'); plt.semilogx(t, b, '--', color=l.get_color(), label=name + ' can side')
plt.xlabel('time after onset [s]'); plt.ylabel('K per W of short'); plt.legend(); plt.grid(alpha=.3); plt.show()

## Electrode sheet coupled to the 3-D grid
`python sheet.py` runs a 2C discharge with and without a 3 Ω short (about 7 min). The local model is a placeholder; MT-TDFN plugs in through the same interface.

In [ ]:
!python -c "import json, sheet; print(json.dumps(sheet.foil_resistance_check(), indent=1))"